In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install numpy pandas scikit-learn tqdm --quiet


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.utils.weight_norm as weight_norm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from google.colab import drive


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

drive.mount('/content/drive')


In [ ]:
# Change only this version to test a different saved model
EXPERIMENT_VERSION = 'v4'
CHECKPOINT_FILENAME = 'best_model.pt'

PROJECT_DIR = Path('/content/drive/MyDrive/crypto_trading_project')
DATA_DIR = PROJECT_DIR / f'datasets_{EXPERIMENT_VERSION}'
CHECKPOINT_DIR = PROJECT_DIR / f'checkpoints_{EXPERIMENT_VERSION}'

TEST_PATH = DATA_DIR / f'BTCUSDT_15m_test_windows_norm_{EXPERIMENT_VERSION}.npz'
CHECKPOINT_PATH = CHECKPOINT_DIR / CHECKPOINT_FILENAME

print('Experiment version:', EXPERIMENT_VERSION)
print('Test dataset:', TEST_PATH)
print('Checkpoint path:', CHECKPOINT_PATH)

if not TEST_PATH.exists():
    raise FileNotFoundError(f'Test dataset not found: {TEST_PATH}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')


In [ ]:
test_data = np.load(TEST_PATH)
X_test = test_data['X']
y_test = test_data['y']
test_time = test_data['end_time']

print('X_test shape:', X_test.shape)
print('y_test shape:', y_test.shape)
print('First timestamp:', test_time[0])
print('Last timestamp:', test_time[-1])


In [ ]:
class CryptoSequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32).transpose(0, 1)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y


class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = weight_norm(nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2
        )
        self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None
        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        residual = x if self.downsample is None else self.downsample(x)
        return self.final_relu(out + residual)


class TCNClassifier(nn.Module):
    def __init__(self, input_channels, channels, kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_channels = input_channels

        for i, out_channels in enumerate(channels):
            dilation = 2 ** i
            layers.append(
                TemporalBlock(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_channels = out_channels

        self.tcn = nn.Sequential(*layers)
        self.classifier = nn.Linear(channels[-1], 1)

    def forward(self, x):
        features = self.tcn(x)
        last_timestep = features[:, :, -1]
        logits = self.classifier(last_timestep).squeeze(-1)
        return logits


In [ ]:
def compute_classification_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
    except ValueError:
        metrics['roc_auc'] = float('nan')
    return metrics


@torch.no_grad()
def evaluate_test_set(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_probs = []
    all_targets = []

    for x_batch, y_batch in tqdm(loader, desc='Testing'):
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        probs = torch.sigmoid(logits)

        running_loss += loss.item() * x_batch.size(0)
        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

    test_loss = running_loss / len(loader.dataset)
    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)
    metrics = compute_classification_metrics(all_targets, all_probs)
    metrics['loss'] = test_loss
    return metrics


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
checkpoint_config = checkpoint.get('config', {})

input_channels = checkpoint_config.get('input_channels', X_test.shape[2])
channels = checkpoint_config.get('channels', [32, 64, 64, 64, 64, 64])
kernel_size = checkpoint_config.get('kernel_size', 3)
dropout = checkpoint_config.get('dropout', 0.3)
batch_size = checkpoint_config.get('batch_size', 128)

model = TCNClassifier(
    input_channels=input_channels,
    channels=channels,
    kernel_size=kernel_size,
    dropout=dropout,
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

test_dataset = CryptoSequenceDataset(X_test, y_test)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
criterion = nn.BCEWithLogitsLoss()

best_epoch = checkpoint.get('epoch', 0)
best_val_loss = checkpoint.get('best_val_loss', float('inf'))

print('Loaded checkpoint config:', checkpoint_config)
print(f'Loaded best model from epoch {best_epoch}')
print(f'Best validation loss: {best_val_loss:.6f}')
print('Test loader batch size:', batch_size)


In [ ]:
test_metrics = evaluate_test_set(model, test_loader, criterion, device)

print('\nTEST RESULTS')
print('-' * 40)
print(f"Loss:      {test_metrics['loss']:.6f}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1:        {test_metrics['f1']:.4f}")
print(f"ROC-AUC:   {test_metrics['roc_auc']:.4f}")


In [ ]:
VERSIONS_TO_COMPARE = ['v1', 'v2', 'v3', 'v4', 'v5']


def evaluate_saved_version(version: str) -> dict:
    data_dir = PROJECT_DIR / f'datasets_{version}'
    checkpoint_dir = PROJECT_DIR / f'checkpoints_{version}'
    test_path = data_dir / f'BTCUSDT_15m_test_windows_norm_{version}.npz'
    checkpoint_path = checkpoint_dir / CHECKPOINT_FILENAME

    if not test_path.exists():
        raise FileNotFoundError(f'Test dataset not found: {test_path}')
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    test_data = np.load(test_path)
    x_test = test_data['X']
    y_test_local = test_data['y']

    checkpoint_local = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config_local = checkpoint_local.get('config', {})

    model_local = TCNClassifier(
        input_channels=config_local.get('input_channels', x_test.shape[2]),
        channels=config_local.get('channels', [32, 64, 64, 64, 64, 64]),
        kernel_size=config_local.get('kernel_size', 3),
        dropout=config_local.get('dropout', 0.3),
    ).to(device)
    model_local.load_state_dict(checkpoint_local['model_state_dict'])

    batch_size_local = config_local.get('batch_size', 128)
    test_dataset_local = CryptoSequenceDataset(x_test, y_test_local)
    test_loader_local = DataLoader(
        test_dataset_local,
        batch_size=batch_size_local,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    criterion_local = nn.BCEWithLogitsLoss()
    metrics_local = evaluate_test_set(model_local, test_loader_local, criterion_local, device)

    return {
        'version': version,
        'status': 'ok',
        'epoch': checkpoint_local.get('epoch', 0),
        'best_val_loss': checkpoint_local.get('best_val_loss', float('nan')),
        'test_rows': int(len(y_test_local)),
        'loss': metrics_local['loss'],
        'accuracy': metrics_local['accuracy'],
        'precision': metrics_local['precision'],
        'recall': metrics_local['recall'],
        'f1': metrics_local['f1'],
        'roc_auc': metrics_local['roc_auc'],
    }


In [ ]:
comparison_rows = []

for version in VERSIONS_TO_COMPARE:
    print(f'Running evaluation for {version}...')
    try:
        comparison_rows.append(evaluate_saved_version(version))
    except Exception as exc:
        comparison_rows.append({
            'version': version,
            'status': f'error: {exc}',
        })

comparison_df = pd.DataFrame(comparison_rows)
metric_columns = ['best_val_loss', 'loss', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']
available_metric_columns = [col for col in metric_columns if col in comparison_df.columns]
if available_metric_columns:
    comparison_df[available_metric_columns] = comparison_df[available_metric_columns].round(4)

comparison_df
